# AI Coach — POC: Import Garmin Activities via Garmin Connect API

Credentials werden sicher per Eingabe abgefragt. Die Aktivitäten werden direkt von
Garmin Connect heruntergeladen und als `.fit` Dateien in `data/activities/` gespeichert.

In [1]:
import getpass
import io
import zipfile
from pathlib import Path
import fitparse
import pandas as pd
from garminconnect import Garmin

In [2]:
import ssl
import os
import certifi
import requests
from requests.adapters import HTTPAdapter


class _GarminTLSAdapter(HTTPAdapter):
    """HTTPAdapter mit ssl.create_default_context() für Python 3.14-Kompatibilität.

    Root-Cause: ssl_wrap_socket ruft load_verify_locations(certifi) auf dem
    ssl.create_default_context()-Kontext auf (weil cert_verify conn.ca_certs
    setzt), was den TLS-Handshake mit Garmin-Servern zerstört.
    Fix: cert_verify überschreiben → ca_certs/ca_cert_dir NIE setzen.
    Unser Kontext hat bereits System-Certs + certifi geladen.
    """

    def __init__(self, **kwargs):
        ctx = ssl.create_default_context()          # Windows-Zertifikatsspeicher
        ctx.load_verify_locations(certifi.where())  # certifi vorab laden
        self._ssl_context = ctx
        super().__init__(**kwargs)

    def init_poolmanager(self, *args, **kwargs):
        kwargs["ssl_context"] = self._ssl_context
        return super().init_poolmanager(*args, **kwargs)

    def proxy_manager_for(self, proxy, **proxy_kwargs):
        proxy_kwargs["ssl_context"] = self._ssl_context
        return super().proxy_manager_for(proxy, **proxy_kwargs)

    def build_connection_pool_key_attributes(self, request, verify, cert=None):
        host_params, pool_kwargs = super().build_connection_pool_key_attributes(
            request, verify, cert
        )
        pool_kwargs["ssl_context"] = self._ssl_context
        return host_params, pool_kwargs

    def cert_verify(self, conn, url, verify, cert):
        """cert_reqs setzen, aber KEIN ca_certs/ca_cert_dir.

        Verhindert, dass ssl_wrap_socket load_verify_locations() auf unserem
        ssl_context aufruft – das zerstört den TLS-Handshake bei Garmin.
        Unser ssl_context hat die CA-Zertifikate bereits korrekt geladen.
        """
        if url.lower().startswith("https") and verify:
            conn.cert_reqs = "CERT_REQUIRED"
        else:
            conn.cert_reqs = "CERT_NONE"
        conn.ca_certs = None
        conn.ca_cert_dir = None
        # Client-Zertifikat setzen wenn angegeben
        if cert:
            if not isinstance(cert, str):
                conn.cert_file = cert[0]
                conn.key_file = cert[1]
            else:
                conn.cert_file = cert
                conn.key_file = None


def apply_garmin_tls_fix(garmin_instance):
    """Ersetzt die Standard-Adapter des Garmin-Clients durch _GarminTLSAdapter."""
    adapter = _GarminTLSAdapter(pool_connections=20, pool_maxsize=20)
    garmin_instance.client._api_session.mount("https://", adapter)
    garmin_instance.client.cs.mount("https://", adapter)
    print("Garmin-TLS-Fix angewendet")


In [3]:
# --- Config ---
ACTIVITIES_DIR = Path("../data/activities")
ACTIVITIES_DIR.mkdir(parents=True, exist_ok=True)
TOKEN_STORE = Path("../data/.garmin_tokens")
NUM_ACTIVITIES = 100

# --- Login (mit Token-Cache) ---
# Beim ersten Aufruf: E-Mail/Passwort eingeben → Token wird gespeichert.
# Ab dem zweiten Aufruf: Token wird automatisch geladen, keine Eingabe nötig.
token_files_exist = TOKEN_STORE.exists() and any(TOKEN_STORE.iterdir()) if TOKEN_STORE.exists() else False

if token_files_exist:
    client = Garmin()
else:
    email = input("Garmin Connect E-Mail: ")
    password = getpass.getpass("Passwort: ")
    client = Garmin(email, password)

# TLS-Fix für Python 3.12+ / Garmin-Server-Inkompatibilität
apply_garmin_tls_fix(client)

client.login(tokenstore=str(TOKEN_STORE))
print("Login erfolgreich.")

# --- FIT-Dateien herunterladen und entpacken ---
activities = client.get_activities(0, NUM_ACTIVITIES)

for act in activities:
    act_id = act["activityId"]
    act_name = act.get("activityName", str(act_id))
    fit_path = ACTIVITIES_DIR / f"{act_id}.fit"
    if not fit_path.exists():
        zip_data = client.download_activity(act_id, dl_fmt=client.ActivityDownloadFormat.ORIGINAL)
        with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
            fit_names = [n for n in zf.namelist() if n.endswith(".fit")]
            if fit_names:
                fit_path.write_bytes(zf.read(fit_names[0]))
                print(f"  Heruntergeladen: {fit_path.name}  ({act_name})")
            else:
                print(f"  Keine .fit in ZIP für Aktivität {act_id} ({act_name})")
    else:
        print(f"  Bereits vorhanden: {fit_path.name}")

fit_files = sorted(ACTIVITIES_DIR.glob("*.fit"))
print(f"\nGesamt: {len(fit_files)} .fit Datei(en) in {ACTIVITIES_DIR.resolve()}")


Garmin-TLS-Fix angewendet
Login erfolgreich.
  Bereits vorhanden: 22746149597.fit
  Bereits vorhanden: 22736420829.fit
  Bereits vorhanden: 22715678237.fit
  Bereits vorhanden: 22691212409.fit
  Bereits vorhanden: 22675360493.fit
  Bereits vorhanden: 22651476092.fit
  Bereits vorhanden: 22631636721.fit
  Bereits vorhanden: 22607649768.fit
  Bereits vorhanden: 22572922875.fit
  Bereits vorhanden: 22563039853.fit
  Bereits vorhanden: 22550078368.fit
  Bereits vorhanden: 22524864188.fit
  Bereits vorhanden: 22524862706.fit
  Bereits vorhanden: 22478650961.fit
  Bereits vorhanden: 22455357310.fit
  Bereits vorhanden: 22418573707.fit
  Bereits vorhanden: 22395566165.fit
  Bereits vorhanden: 22361089765.fit
  Bereits vorhanden: 22330086513.fit

Gesamt: 19 .fit Datei(en) in E:\GoogleDrive\privat\Code_Repo\AI_Coach\data\activities


In [4]:
def parse_fit_file(fit_path: Path) -> pd.DataFrame:
    """Parse a single .fit file.

    Returns a DataFrame of 'record' messages (time-series data) with two
    additional columns 'sport' and 'sub_sport' taken from the 'session' message.
    Unknown / vendor-specific fields (None name or numeric name) are dropped.
    """
    fit = fitparse.FitFile(str(fit_path))

    # --- session meta (sport, sub_sport) ---
    sport = None
    sub_sport = None
    for msg in fit.get_messages("session"):
        for data in msg:
            if data.name == "sport":
                sport = str(data.value) if data.value is not None else None
            elif data.name == "sub_sport":
                sub_sport = str(data.value) if data.value is not None else None
        break  # only first session message needed

    # --- time-series records ---
    rows = []
    for msg in fit.get_messages("record"):
        row = {}
        for data in msg:
            # skip unknown fields: name is None, a plain int, or starts with "unknown"
            if data.name is None:
                continue
            if isinstance(data.name, int):
                continue
            if str(data.name).startswith("unknown"):
                continue
            row[data.name] = data.value
        rows.append(row)

    df = pd.DataFrame(rows)
    df.insert(0, "sport", sport)
    df.insert(1, "sub_sport", sub_sport)
    return df

In [5]:
fit_files[0]

WindowsPath('../data/activities/22330086513.fit')

In [6]:
import getpass
import io
import zipfile
from pathlib import Path
import fitparse
import pandas as pd
from garminconnect import Garmin

# --- Config ---
ACTIVITIES_DIR = Path("../data/activities")
ACTIVITIES_DIR.mkdir(parents=True, exist_ok=True)
TOKEN_STORE = Path("../data/.garmin_tokens")
NUM_ACTIVITIES = 100

# --- Login (mit Token-Cache) ---
client = Garmin()
try:
    client.login(tokenstore=str(TOKEN_STORE))
    print("Login mit gespeichertem Token erfolgreich.")
except Exception:
    email = input("Garmin Connect E-Mail: ")
    password = getpass.getpass("Passwort: ")
    client = Garmin(email, password)
    client.login()
    client.garth.dump(str(TOKEN_STORE))
    print("Login erfolgreich. Token für künftige Sitzungen gespeichert.")

# --- FIT-Dateien herunterladen und entpacken ---
activities = client.get_activities(0, NUM_ACTIVITIES)

for act in activities:
    act_id = act["activityId"]
    act_name = act.get("activityName", str(act_id))
    fit_path = ACTIVITIES_DIR / f"{act_id}.fit"
    if not fit_path.exists():
        zip_data = client.download_activity(act_id, dl_fmt=client.ActivityDownloadFormat.ORIGINAL)
        with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
            fit_names = [n for n in zf.namelist() if n.endswith(".fit")]
            if fit_names:
                fit_path.write_bytes(zf.read(fit_names[0]))
                print(f"  Heruntergeladen: {fit_path.name}  ({act_name})")
            else:
                print(f"  Keine .fit in ZIP für Aktivität {act_id} ({act_name})")
    else:
        print(f"  Bereits vorhanden: {fit_path.name}")

fit_files = sorted(ACTIVITIES_DIR.glob("*.fit"))
print(f"\nGesamt: {len(fit_files)} .fit Datei(en) in {ACTIVITIES_DIR.resolve()}")

Login mit gespeichertem Token erfolgreich.
  Bereits vorhanden: 22746149597.fit
  Bereits vorhanden: 22736420829.fit
  Bereits vorhanden: 22715678237.fit
  Bereits vorhanden: 22691212409.fit
  Bereits vorhanden: 22675360493.fit
  Bereits vorhanden: 22651476092.fit
  Bereits vorhanden: 22631636721.fit
  Bereits vorhanden: 22607649768.fit
  Bereits vorhanden: 22572922875.fit
  Bereits vorhanden: 22563039853.fit
  Bereits vorhanden: 22550078368.fit
  Bereits vorhanden: 22524864188.fit
  Bereits vorhanden: 22524862706.fit
  Bereits vorhanden: 22478650961.fit
  Bereits vorhanden: 22455357310.fit
  Bereits vorhanden: 22418573707.fit
  Bereits vorhanden: 22395566165.fit
  Bereits vorhanden: 22361089765.fit
  Bereits vorhanden: 22330086513.fit

Gesamt: 19 .fit Datei(en) in E:\GoogleDrive\privat\Code_Repo\AI_Coach\data\activities


In [7]:
# Parse the first available file as a quick test
if fit_files:
    df = parse_fit_file(fit_files[0])
    print(f"Parsed: {fit_files[0].name}  →  {len(df)} records, {len(df.columns)} fields")
    print("\nColumns:", list(df.columns))
    df.head()
else:
    print("No .fit files found. Copy files from your Garmin device to:", ACTIVITIES_DIR.resolve())

Parsed: 22330086513.fit  →  146 records, 9 fields

Columns: ['sport', 'sub_sport', 'distance', 'enhanced_altitude', 'heart_rate', 'position_lat', 'position_long', 'timestamp', 'enhanced_speed']


In [8]:
# Parse ALL .fit files into a single combined DataFrame
# activity_meta liefert den offiziellen Garmin-Aktivitätstyp aus der API
activity_meta = {
    str(act["activityId"]): act.get("activityType", {}).get("typeKey", "unknown")
    for act in activities
}

all_dfs = []
for f in fit_files:
    df_tmp = parse_fit_file(f)
    act_id_str = f.stem  # Dateiname ohne .fit = activityId
    fallback = df_tmp["sport"].iloc[0] if len(df_tmp) else "unknown"
    df_tmp = df_tmp.assign(
        activity_type=activity_meta.get(act_id_str, fallback),
        source_file=f.name,
    )
    # activity_type und source_file als erste Spalten
    cols = ["activity_type", "source_file"] + [c for c in df_tmp.columns if c not in ("activity_type", "source_file")]
    all_dfs.append(df_tmp[cols])

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f"Total records: {len(df_all)}")
    print(f"\nAktivitäten:")
    print(df_all.groupby("source_file")["activity_type"].first().to_string())
    print(f"\nSpalten ({len(df_all.columns)}): {list(df_all.columns)}")
    df_all.head()
else:
    print("No .fit files to process.")

Total records: 34280

Aktivitäten:
source_file
22330086513.fit              cycling
22361089765.fit              running
22395566165.fit         lap_swimming
22418573707.fit              running
22455357310.fit              cycling
22478650961.fit    treadmill_running
22524862706.fit       indoor_cycling
22524864188.fit              running
22550078368.fit               hiking
22563039853.fit              running
22572922875.fit              cycling
22607649768.fit              running
22631636721.fit       indoor_cycling
22651476092.fit              cycling
22675360493.fit       indoor_cycling
22691212409.fit              cycling
22715678237.fit              running
22736420829.fit              cycling
22746149597.fit              running

Spalten (22): ['activity_type', 'source_file', 'sport', 'sub_sport', 'distance', 'enhanced_altitude', 'heart_rate', 'position_lat', 'position_long', 'timestamp', 'enhanced_speed', 'cadence', 'fractional_cadence', 'power', 'stance_time', 'stance_time

In [9]:
df_all.head()

,activity_type,source_file,sport,sub_sport,distance,enhanced_altitude,heart_rate,position_lat,position_long,timestamp,...,fractional_cadence,power,stance_time,stance_time_balance,stance_time_percent,step_length,vertical_oscillation,vertical_ratio,accumulated_power,temperature
0,cycling,22330086513.fit,cycling,generic,0.0,104.6,98,NaN,NaN,2026-03-28 15:50:55,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cycling,22330086513.fit,cycling,generic,0.0,104.6,97,NaN,NaN,2026-03-28 15:50:56,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,cycling,22330086513.fit,cycling,generic,0.0,104.6,94,NaN,NaN,2026-03-28 15:51:02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,cycling,22330086513.fit,cycling,generic,0.0,104.6,97,NaN,NaN,2026-03-28 15:51:14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cycling,22330086513.fit,cycling,generic,0.0,104.6,100,NaN,NaN,2026-03-28 15:51:27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Process Training Data

In [10]:
def expand_timestamp(df: pd.DataFrame, col: str = "timestamp") -> pd.DataFrame:
    ts = pd.to_datetime(df[col])
    df = df.copy()
    df["jahr"]    = ts.dt.year
    df["monat"]   = ts.dt.month
    df["tag"]     = ts.dt.day
    df["uhrzeit"] = ts.dt.strftime("%H:%M:%S")
    return df

def reorder_columns(df: pd.DataFrame, first_cols: list) -> pd.DataFrame:
    ordered = first_cols + [c for c in df.columns if c not in first_cols]
    return df[ordered]

In [11]:
df_all = expand_timestamp(df_all)
df_all = reorder_columns(df_all, ["source_file", "timestamp", "jahr", "monat", "tag", "uhrzeit"])

In [18]:
df_all.head()

,source_file,timestamp,jahr,monat,tag,uhrzeit,activity_type,sport,sub_sport,distance,...,position_long,enhanced_speed,cadence,fractional_cadence,power,stance_time,step_length,vertical_oscillation,vertical_ratio,accumulated_power
0,22330086513.fit,2026-03-28 15:50:55,2026,3,28,15:50:55,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22330086513.fit,2026-03-28 15:50:56,2026,3,28,15:50:56,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,22330086513.fit,2026-03-28 15:51:02,2026,3,28,15:51:02,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,22330086513.fit,2026-03-28 15:51:14,2026,3,28,15:51:14,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,22330086513.fit,2026-03-28 15:51:27,2026,3,28,15:51:27,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Analyze one specific training event

In [13]:
df_all.groupby("sub_sport").count()

,source_file,timestamp,jahr,monat,tag,uhrzeit,activity_type,sport,distance,enhanced_altitude,...,fractional_cadence,power,stance_time,stance_time_balance,stance_time_percent,step_length,vertical_oscillation,vertical_ratio,accumulated_power,temperature
sub_sport,,,,,,,,,,,,,,,,,,,,,
generic,25677,25677,25677,25677,25677,25677,25677,25677,25677,25677,...,21787,20606,9234,0,0,13187,17532,13187,20489,0
indoor_cycling,2095,2095,2095,2095,2095,2095,2095,2095,2095,0,...,0,0,0,0,0,0,0,0,0,0
lap_swimming,1058,1058,1058,1058,1058,1058,1058,1058,0,0,...,0,0,0,0,0,0,0,0,0,0
treadmill,5450,5450,5450,5450,5450,5450,5450,5450,5450,0,...,5450,5449,84,0,0,4102,4728,4102,5446,0


In [14]:
def missing_data_report(df: pd.DataFrame) -> pd.DataFrame:
    total = len(df)
    missing = df.isna().sum()
    return pd.DataFrame({
        "fehlend":   missing,
        "anteil_%":  (missing / total * 100).round(2)
    }).sort_values("anteil_%", ascending=False)



In [15]:
missing_data_report(df_all)

,fehlend,anteil_%
temperature,34280,100.00
stance_time_percent,34280,100.00
stance_time_balance,34280,100.00
stance_time,24962,72.82
step_length,16991,49.57
vertical_ratio,16991,49.57
vertical_oscillation,12020,35.06
position_long,8659,25.26
position_lat,8659,25.26
enhanced_altitude,8603,25.10


In [16]:
def drop_high_missing(df: pd.DataFrame, threshold: float = 50.0) -> pd.DataFrame:
    missing_pct = df.isna().mean() * 100
    keep = missing_pct[missing_pct <= threshold].index
    dropped = missing_pct[missing_pct > threshold].index.tolist()
    print(f"Gelöschte Spalten ({len(dropped)}): {dropped}")
    return df[keep]



In [17]:
df_all = drop_high_missing(df_all, threshold=90.0)

Gelöschte Spalten (3): ['stance_time_balance', 'stance_time_percent', 'temperature']


In [20]:
df_all.head()

,source_file,timestamp,jahr,monat,tag,uhrzeit,activity_type,sport,sub_sport,distance,...,position_long,enhanced_speed,cadence,fractional_cadence,power,stance_time,step_length,vertical_oscillation,vertical_ratio,accumulated_power
0,22330086513.fit,2026-03-28 15:50:55,2026,3,28,15:50:55,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22330086513.fit,2026-03-28 15:50:56,2026,3,28,15:50:56,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,22330086513.fit,2026-03-28 15:51:02,2026,3,28,15:51:02,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,22330086513.fit,2026-03-28 15:51:14,2026,3,28,15:51:14,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,22330086513.fit,2026-03-28 15:51:27,2026,3,28,15:51:27,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
